In [ ]:
import pandas as pd

df = pd.read_csv("Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")
print(df.shape)
df.head()

(26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [3]:
df.info()
print(df.isnull().sum())
print("Duplicated rows:", df.duplicated().sum())
print("Duplicated instructions:", df["instruction"].duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   flags        26872 non-null  str  
 1   instruction  26872 non-null  str  
 2   category     26872 non-null  str  
 3   intent       26872 non-null  str  
 4   response     26872 non-null  str  
dtypes: str(5)
memory usage: 1.0 MB
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64
Duplicated rows: 0
Duplicated instructions: 2237


In [4]:
dup_mask = df["instruction"].duplicated(keep=False)
dups = df[dup_mask]

conflicts = dups.groupby("instruction")["intent"].nunique()
print("Duplicated instruction groups:", conflicts.shape[0])
print("Groups with conflicting intents:", (conflicts > 1).sum())

Duplicated instruction groups: 989
Groups with conflicting intents: 0


In [5]:
import re

def clean_text(text):
    # 1) توحيد الـ placeholders: {{Order Number}} -> ph_order_number
    text = re.sub(
        r"\{\{(.*?)\}\}",
        lambda m: "ph_" + re.sub(r"\s+", "_", m.group(1).strip().lower()),
        text,
    )
    # 2) lowercase
    text = text.lower()
    # 3) تنضيف المسافات الزيادة
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["instruction_clean"] = df["instruction"].apply(clean_text)

df[["instruction", "instruction_clean"]].head(8)

,instruction,instruction_clean
0,question about cancelling order {{Order Number}},question about cancelling order ph_order_number
1,i have a question about cancelling oorder {{Or...,i have a question about cancelling oorder ph_o...
2,i need help cancelling puchase {{Order Number}},i need help cancelling puchase ph_order_number
3,I need to cancel purchase {{Order Number}},i need to cancel purchase ph_order_number
4,"I cannot afford this order, cancel purchase {{...","i cannot afford this order, cancel purchase ph..."
5,can you help me cancel order {{Order Number}}?,can you help me cancel order ph_order_number?
6,"I can no longer afford order {{Order Number}},...","i can no longer afford order ph_order_number, ..."
7,I am trying to cancel purchase {{Order Number}},i am trying to cancel purchase ph_order_number


In [6]:
before = len(df)
df_dedup = df.drop_duplicates(subset="instruction_clean", keep="first").reset_index(drop=True)

print("Before:", before)
print("After :", len(df_dedup))
print("Removed:", before - len(df_dedup))

Before: 26872
After : 24274
Removed: 2598


In [7]:
df_dedup["intent"].value_counts().describe()

count     27.000000
mean     899.037037
std      133.974303
min      436.000000
25%      865.500000
50%      961.000000
75%      991.000000
max      997.000000
Name: count, dtype: float64

In [8]:
before = df["intent"].value_counts()
after = df_dedup["intent"].value_counts()

comp = pd.DataFrame({"before": before, "after": after})
comp["removed"] = comp["before"] - comp["after"]
comp.sort_values("removed", ascending=False).head(8)

,before,after,removed
intent,,,
cancel_order,998,436,562
delivery_options,995,655,340
track_refund,998,692,306
edit_account,1000,766,234
track_order,995,783,212
switch_account,1000,836,164
change_order,997,856,141
create_account,997,875,122


In [9]:
from sklearn.model_selection import train_test_split
import os

train, temp = train_test_split(
    df_dedup, test_size=0.2, stratify=df_dedup["intent"], random_state=42
)
val, test = train_test_split(
    temp, test_size=0.5, stratify=temp["intent"], random_state=42
)

print("Train:", len(train), "| Val:", len(val), "| Test:", len(test))

os.makedirs("processed_data", exist_ok=True)
train.to_csv("processed_data/train.csv", index=False)
val.to_csv("processed_data/val.csv", index=False)
test.to_csv("processed_data/test.csv", index=False)

Train: 19419 | Val: 2427 | Test: 2428


In [10]:
print(train["intent"].value_counts(normalize=True).head(3))
print(test["intent"].value_counts(normalize=True).head(3))

intent
complaint                0.041042
place_order              0.040888
registration_problems    0.040888
Name: proportion, dtype: float64
intent
review                   0.041186
check_payment_methods    0.041186
payment_issue            0.041186
Name: proportion, dtype: float64


In [11]:
train_set = set(train["instruction_clean"])
print("Train/Test overlap:", len(train_set & set(test["instruction_clean"])))
print("Train/Val overlap :", len(train_set & set(val["instruction_clean"])))

Train/Test overlap: 0
Train/Val overlap : 0


In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_csv("processed_data/train.csv")
val = pd.read_csv("processed_data/val.csv")

# كلمات نفي وأسئلة ممكن تفرق في المعنى، فنحافظ عليها
keep = {"not", "no", "nor", "cannot", "can", "how", "what", "where", "when", "why", "who"}
custom_stop = list(ENGLISH_STOP_WORDS - keep)

def run(stop_words, name):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words=stop_words)
    X_train = vec.fit_transform(train["instruction_clean"])
    X_val = vec.transform(val["instruction_clean"])

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, train["intent"])
    pred = clf.predict(X_val)

    return {
        "setting": name,
        "n_features": X_train.shape[1],
        "accuracy": round(accuracy_score(val["intent"], pred), 4),
        "macro_f1": round(f1_score(val["intent"], pred, average="macro"), 4),
    }

results = pd.DataFrame([
    run(None, "No stop-word removal"),
    run("english", "sklearn english stop words"),
    run(custom_stop, "Custom (keep negation + question words)"),
])
results

,setting,n_features,accuracy,macro_f1
0,No stop-word removal,4974,0.9926,0.9912
1,sklearn english stop words,4897,0.9753,0.9737
2,Custom (keep negation + question words),5310,0.9757,0.9743


setting	n_features	accuracy	macro_f1
0	No stop-word removal	4974	0.9926	0.9912
1	sklearn english stop words	4897	0.9753	0.9737
2	Custom (keep negation + question words)	5310	0.9757	0.9743
